In [30]:
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
import pickle
import json
import pickle
import re
from pydantic import BaseModel

In [6]:
openai_client = OpenAI()

In [7]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [8]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [10]:
def build_prompt(question, search_results):
    search_json = json.dumps(search_results)
    return prompt_template.format(
        question=question,
        context=search_json
    )

In [11]:
# retries documents relevant to the input question
def rag(question):
    search_results = search(question) # retrieve relevant documents
    user_prompt = build_prompt(question, search_results) # combine search results and user query
    results = llm(user_prompt, instructions=instructions)
    return results

In [14]:
video_id = 'ph1PxZIkz1o'

!wget https://github.com/alexeygrigorev/ai-bootcamp-codespace/raw/refs/heads/main/week1/ph1PxZIkz1o.bin

# ytt_api = YouTubeTranscriptApi()
# transcript = ytt_api.fetch(video_id)

# with open(f'{video_id}.bin', 'wb') as f_out:
#     pickle.dump(transcript, f_out)

--2025-10-24 08:31:12--  https://github.com/alexeygrigorev/ai-bootcamp-codespace/raw/refs/heads/main/week1/ph1PxZIkz1o.bin
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/alexeygrigorev/ai-bootcamp-codespace/refs/heads/main/week1/ph1PxZIkz1o.bin [following]
--2025-10-24 08:31:12--  https://raw.githubusercontent.com/alexeygrigorev/ai-bootcamp-codespace/refs/heads/main/week1/ph1PxZIkz1o.bin
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 101591 (99K) [application/octet-stream]
Saving to: ‘ph1PxZIkz1o.bin’

ph1PxZIkz1o.bin     100%[===================>]  99.21K  --.-KB/s    in 0.003s  

2025-10-24 

In [15]:
with open(f'{video_id}.bin', 'rb') as f_in:
    transcript = pickle.load(f_in)

In [16]:
transcript[:10]

[FetchedTranscriptSnippet(text='So hi everyone. Uh today we are going to', start=0.0, duration=5.04),
 FetchedTranscriptSnippet(text='talk about our upcoming course. The', start=2.96, duration=3.52),
 FetchedTranscriptSnippet(text='upcoming course is called machine', start=5.04, duration=5.92),
 FetchedTranscriptSnippet(text='learning zoom camp. And um this is', start=6.48, duration=5.92),
 FetchedTranscriptSnippet(text='already I put the link in the', start=10.96, duration=3.599),
 FetchedTranscriptSnippet(text="description. So if you're watching um", start=12.4, duration=4.719),
 FetchedTranscriptSnippet(text="this video in recording or you're", start=14.559, duration=4.88),
 FetchedTranscriptSnippet(text='watching it live, you go here in the', start=17.119, duration=4.561),
 FetchedTranscriptSnippet(text='description after under this video and', start=19.439, duration=5.6),
 FetchedTranscriptSnippet(text='then you see a link course. uh click on', start=21.68, duration=6.24)]

In [ ]:
def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

In [ ]:
def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)

In [19]:
subtitles = make_subtitles(transcript)

In [22]:
print(subtitles[:500])

0:00 So hi everyone. Uh today we are going to
0:02 talk about our upcoming course. The
0:05 upcoming course is called machine
0:06 learning zoom camp. And um this is
0:10 already I put the link in the
0:12 description. So if you're watching um
0:14 this video in recording or you're
0:17 watching it live, you go here in the
0:19 description after under this video and
0:21 then you see a link course. uh click on
0:25 that link and this bring you will bring
0:27 you to
0:29 this website this GitHub


In [23]:
instructions = """
    Summarize the transcript and describe the main purpose of the video
    and the main ideas. 

    Also output chapters with time. Use usual sentence case, not Title Case for the chapter.

    Output format: 

    <OUTPUT>
    Summary

    timestamp chapter 
    timestamp chapter
    ...
    timestamp chapter
    </OUTPUT>
""".strip()

In [ ]:
answer = llm(subtitles, instructions=instructions)

<OUTPUT>
**Summary**

In this video, the speaker discusses the details of an upcoming course titled "Machine Learning Zoom Camp." The course is set to start on September 15th and aims to equip learners with essential skills for machine learning, focusing on engineering aspects rather than just theoretical knowledge. The speaker encourages viewers to ask questions via an external link and mentions the course is now in its fifth edition. Many past participants have successfully found jobs after completing it, and while job placements aren't guaranteed, the skills taught significantly enhance employability.

The course covers ten modules, four of which will be updated for this edition. Topics include fundamental machine learning techniques, deployment strategies, and basics of deep learning but will not delve deeply into specific areas like computer vision. Prerequisites include a basic understanding of programming, especially in Python, as well as familiarity with command-line interfaces

In [26]:
def strip_matching_outer_html_tags(text: str) -> str:
    match = re.match(r"^\s*<(\w+)[^>]*>\s*(.*?)\s*</\1>\s*$", text, re.DOTALL)
    if match:
        return match.group(2).strip()
    return text.strip()

answer = strip_matching_outer_html_tags(answer)

## Format LLM output

In [31]:
class Chapter(BaseModel):
    timestamp: str
    title: str

class YTSummaryResponse(BaseModel):
    summary: str
    chapters: list[Chapter]

In [ ]:
instructions = """
    Summarize the transcript and describe the main purpose of the video
    and the main ideas. 

    Also output chapters with time. Use usual sentence case, not Title Case for the chapter.

    More chapters is better than fewer chapters. Have a chapter at least every 3-5 minutes
""".strip()

In [34]:
def llm_structured(instructions, user_prompt, output_format, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_format
    )

    # return response.output[0].content[0].parsed
    return response.output_parsed

In [35]:
summary = llm_structured(
    instructions=instructions,
    user_prompt=subtitles,
    output_format=YTSummaryResponse
)

In [37]:
print(summary.summary)
print(summary.chapters)

The video introduces the upcoming "Machine Learning Zoom Camp" course, focusing on its structure, prerequisites, content updates, and Q&A for potential participants. It emphasizes that the course aims to equip learners with essential skills for machine learning engineering, rather than data science. Expectations for job placement, updates to course material, and the balance between learning theory and practical applications are discussed.
[Chapter(timestamp='0:00', title='Introduction to the course'), Chapter(timestamp='2:38', title='Course content and updates'), Chapter(timestamp='5:05', title='Questions about job placement'), Chapter(timestamp='8:18', title='Preparation and prerequisites'), Chapter(timestamp='11:06', title="What skills you'll gain"), Chapter(timestamp='13:43', title='Recommended materials and resources'), Chapter(timestamp='16:19', title='Workstation requirements for the course'), Chapter(timestamp='19:06', title='Using AI tools like ChatGPT for learning'), Chapter(t